# POC Detector Shelf Detection (Colab atau lokal)

Notebook ini membuktikan bahwa `best.pt` hasil training dapat mendeteksi produk pada foto rak. POC mencakup inferensi satu gambar, visualisasi bounding box, ringkasan confidence dan latency, penyimpanan JSON, serta evaluasi test set secara opsional.

POC memuat model langsung dengan Ultralytics. Cara ini sengaja tidak menggunakan wrapper ONNX atau compliance engine repo, sehingga hasil detector dapat dinilai secara terpisah.

## 1. Konfigurasi POC

Nilai yang paling sering diubah diletakkan di satu cell:

- `MODEL_PATH`: model yang dihasilkan notebook training.
- `INPUT_MODE`: gunakan `upload` untuk memilih gambar dari komputer di Colab, atau `path` untuk membaca gambar dari path biasa.
- `CONFIDENCE`: batas minimum confidence detection.
- `RUN_SKU110K_TEST_EVALUATION`: aktifkan untuk metrik kuantitatif pada SKU-110K test split.
- `DOWNLOAD_PUBLIC_FIELD_SOURCES`: aktifkan untuk mengunduh sumber evaluasi lapangan.
- `RUN_FIELD_SOURCE_EVALUATION`: aktifkan untuk inferensi kualitatif pada sumber lapangan.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
from datetime import datetime, timezone

def detect_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


IS_COLAB = detect_colab()
LOCAL_PROJECT_ROOT = Path.cwd().resolve()
for candidate in (LOCAL_PROJECT_ROOT, *LOCAL_PROJECT_ROOT.parents):
    if (candidate / ".git").is_dir() and (candidate / "notebooks").is_dir():
        LOCAL_PROJECT_ROOT = candidate
        break
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive/shelf-detection")
DRIVE_ROOT = COLAB_DRIVE_ROOT if IS_COLAB else LOCAL_PROJECT_ROOT
MODEL_PATH = DRIVE_ROOT / "models/best.pt"
OUTPUT_DIR = DRIVE_ROOT / "poc_outputs"
FIELD_DATA_DIR = DRIVE_ROOT / "data/test_pairs/public"
FIELD_OUTPUT_DIR = OUTPUT_DIR / "field_sources"
DATASET_NAME = "SKU-110K.yaml"
PYTORCH_INSTALL_ARGS = ["torch", "torchvision", "torchaudio"]
# Untuk Linux + NVIDIA/CUDA, ganti ke wheel resmi PyTorch, misalnya:
# PYTORCH_INSTALL_ARGS = ["torch", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu128"]

INPUT_MODE = "upload" if IS_COLAB else "path"  # Pilihan: "upload", "drive", atau "path"
DRIVE_IMAGE_PATH = DRIVE_ROOT / "poc_images/shelf.jpg"
LOCAL_INPUT_IMAGE_PATH = LOCAL_PROJECT_ROOT / "poc_images/shelf.jpg"
CONFIDENCE = 0.25
IOU_THRESHOLD = 0.45
IMAGE_SIZE = 640
RUN_SKU110K_TEST_EVALUATION = False
DOWNLOAD_PUBLIC_FIELD_SOURCES = False
RUN_FIELD_SOURCE_EVALUATION = False
INCLUDE_GROCERY_DATASET = True
FIELD_MAX_IMAGES = 0  # 0 berarti semua image yang berhasil dikumpulkan

if INPUT_MODE not in {"upload", "drive", "path"}:
    raise ValueError("INPUT_MODE harus 'upload', 'drive', atau 'path'.")
if INPUT_MODE == "upload" and not IS_COLAB:
    raise ValueError("Mode upload hanya tersedia di Google Colab. Gunakan INPUT_MODE='path' di luar Colab.")

print(f"Runtime    : {'Google Colab' if IS_COLAB else 'local / non-Colab'}")
print(f"Root       : {DRIVE_ROOT}")
print(f"Model      : {MODEL_PATH}")
print(f"Input mode : {INPUT_MODE}")
print(f"Confidence : {CONFIDENCE}")

## 2. Siapkan storage persisten

Di Google Colab, cell ini mount Google Drive. Di luar Colab, notebook memakai folder project lokal sebagai root persisten.

In [ ]:
if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Mode non-Colab: memakai folder project lokal sebagai storage persisten.")

for directory in (OUTPUT_DIR, FIELD_OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        f"Model tidak ditemukan: {MODEL_PATH}. Jalankan notebook training terlebih dahulu."
    )

model_sha256 = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
print(f"Model ditemukan ({MODEL_PATH.stat().st_size / 1024**2:.1f} MB)")
print(f"SHA-256: {model_sha256}")

## 3. Instal dependency dan pilih device

POC memasang versi dependency yang sama dengan baseline training detector, ditambah OpenCV, Matplotlib, dan Requests untuk mengunduh sumber evaluasi lapangan. GPU digunakan jika tersedia; POC tetap dapat berjalan di CPU untuk satu gambar, meskipun latency-nya lebih tinggi.

In [ ]:
try:
    import torch
except ModuleNotFoundError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *PYTORCH_INSTALL_ARGS],
        check=True,
    )
    import torch

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ultralytics>=8.4.70",
        "opencv-python-headless>=4.8",
        "matplotlib>=3.7",
        "requests>=2.31",
    ],
    check=True,
)

import cv2
import matplotlib.pyplot as plt
import requests
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 4. Pilih foto rak

Pada mode `upload`, Colab membuka pemilih file dari komputer. Pada mode `drive`, notebook menggunakan `DRIVE_IMAGE_PATH`. Pada mode `path`, notebook menggunakan `LOCAL_INPUT_IMAGE_PATH`, cocok untuk Jupyter lokal, Kaggle, VM, atau server biasa.

In [ ]:
ALLOWED_IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp"}

if INPUT_MODE == "upload":
    from google.colab import files

    uploaded = files.upload()
    uploaded_images = [
        Path(name) for name in uploaded if Path(name).suffix.lower() in ALLOWED_IMAGE_SUFFIXES
    ]
    if not uploaded_images:
        raise RuntimeError("Tidak ada file gambar yang valid pada hasil upload.")
    image_path = uploaded_images[0]
elif INPUT_MODE == "drive":
    image_path = DRIVE_IMAGE_PATH
else:
    image_path = LOCAL_INPUT_IMAGE_PATH

if not image_path.is_file():
    raise FileNotFoundError(f"Gambar tidak ditemukan: {image_path}")
if image_path.suffix.lower() not in ALLOWED_IMAGE_SUFFIXES:
    raise ValueError(f"Format gambar tidak didukung: {image_path.suffix}")

image = cv2.imread(str(image_path))
if image is None:
    raise RuntimeError(f"OpenCV gagal membaca gambar: {image_path}")

height, width = image.shape[:2]
print(f"Gambar: {image_path} ({width}x{height})")

## 5. Muat model dan jalankan inferensi

`YOLO(MODEL_PATH)` memuat checkpoint PyTorch. `model.predict()` melakukan preprocessing, inferensi, non-maximum suppression, dan mengembalikan bounding box.

- `conf` membuang detection dengan confidence rendah.
- `iou` mengendalikan penggabungan box yang saling bertumpuk.
- `imgsz` menentukan resolusi input model.
- `wall_clock_ms` mengukur waktu total pemanggilan dari notebook.

In [ ]:
model = YOLO(str(MODEL_PATH))

started = time.perf_counter()
results = model.predict(
    source=str(image_path),
    conf=CONFIDENCE,
    iou=IOU_THRESHOLD,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    verbose=False,
)
wall_clock_ms = (time.perf_counter() - started) * 1000

if not results:
    raise RuntimeError("Ultralytics tidak mengembalikan hasil inferensi.")

result = results[0]
detection_count = len(result.boxes) if result.boxes is not None else 0
print(f"Jumlah detection: {detection_count}")
print(f"Total latency   : {wall_clock_ms:.1f} ms")
print(f"YOLO speed      : {result.speed}")

## 6. Tampilkan dan simpan visualisasi

`result.plot()` menggambar bounding box serta confidence. Ultralytics menghasilkan array BGR, sehingga warna dikonversi ke RGB hanya untuk ditampilkan oleh Matplotlib. File asli tetap disimpan sebagai JPEG BGR melalui OpenCV.

In [ ]:
annotated = result.plot()
output_image_path = OUTPUT_DIR / f"{image_path.stem}_annotated.jpg"
if not cv2.imwrite(str(output_image_path), annotated):
    raise RuntimeError(f"Gagal menyimpan hasil ke {output_image_path}")

plt.figure(figsize=(16, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f"Shelf Detection POC - {detection_count} products - {wall_clock_ms:.1f} ms")
plt.axis("off")
plt.show()

print(f"Gambar anotasi disimpan: {output_image_path}")

## 7. Simpan hasil terstruktur sebagai JSON

Setiap detection disimpan dengan koordinat `xyxy`, confidence, class ID, dan nama class. Laporan juga mencatat checksum model, parameter inferensi, ukuran gambar, serta rincian latency agar POC dapat direproduksi.

In [ ]:
detections = []
if result.boxes is not None:
    boxes_xyxy = result.boxes.xyxy.cpu().tolist()
    confidences = result.boxes.conf.cpu().tolist()
    class_ids = result.boxes.cls.cpu().tolist()
    for bbox, confidence, class_id_value in zip(boxes_xyxy, confidences, class_ids):
        class_id = int(class_id_value)
        detections.append(
            {
                "bbox_xyxy": [round(float(value), 2) for value in bbox],
                "confidence": round(float(confidence), 6),
                "class_id": class_id,
                "class_name": result.names.get(class_id, str(class_id)),
            }
        )

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_path": str(MODEL_PATH),
    "model_sha256": model_sha256,
    "image_path": str(image_path),
    "image_width": width,
    "image_height": height,
    "confidence_threshold": CONFIDENCE,
    "iou_threshold": IOU_THRESHOLD,
    "image_size": IMAGE_SIZE,
    "device": str(DEVICE),
    "wall_clock_ms": round(wall_clock_ms, 3),
    "yolo_speed_ms": {key: round(float(value), 3) for key, value in result.speed.items()},
    "detection_count": detection_count,
    "detections": detections,
}

output_json_path = OUTPUT_DIR / f"{image_path.stem}_result.json"
output_json_path.write_text(json.dumps(report, indent=2))
print(f"Laporan JSON disimpan: {output_json_path}")
print(json.dumps(report, indent=2)[:3000])

## 8. Evaluasi SKU-110K test set (opsional)

Bagian ini hanya berjalan jika `RUN_SKU110K_TEST_EVALUATION = True`. Ultralytics mengunduh dan menyiapkan SKU-110K resmi secara otomatis bila belum tersedia, lalu `model.val()` menghitung mAP@50, mAP@50-95, precision, dan recall pada split test.

Gunakan bagian ini setelah inferensi satu gambar berhasil. Evaluasi seluruh test set dapat memakan waktu dan kuota GPU cukup besar.

In [ ]:
if RUN_SKU110K_TEST_EVALUATION:
    metrics = model.val(
        data=DATASET_NAME,
        split="test",
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        project=str(OUTPUT_DIR),
        name="sku110k_test_set_evaluation",
        exist_ok=True,
        verbose=True,
    )
    evaluation_metrics = {
        key: round(float(value), 6)
        for key, value in metrics.results_dict.items()
        if isinstance(value, (int, float))
    }
    evaluation_report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_sha256": model_sha256,
        "dataset": DATASET_NAME,
        "split": "test",
        "metrics": evaluation_metrics,
    }
    evaluation_output = OUTPUT_DIR / "sku110k_test_metrics.json"
    evaluation_output.write_text(json.dumps(evaluation_report, indent=2))
    print(json.dumps(evaluation_report, indent=2))
    print(f"Metrik disimpan: {evaluation_output}")
else:
    print("Evaluasi SKU-110K test set dilewati. Ubah RUN_SKU110K_TEST_EVALUATION=True untuk menjalankannya.")

## 9. Unduh sumber evaluasi lapangan (opsional)

Bagian ini mengikuti sumber evaluasi publik pada repo referensi. Script aktif repo referensi mengunduh Grocery Planogram Control Dataset dan Wikimedia Commons. Grocery Dataset juga disebut pada README/docstring repo tersebut, jadi notebook ini menyediakannya sebagai opsi tambahan lewat `INCLUDE_GROCERY_DATASET`. Sumber ini tidak menggantikan SKU-110K untuk training; ia dipakai untuk POC kualitatif pada foto rak nyata.

Catatan lisensi: Grocery Dataset menyatakan **research only, not commercial use**. Grocery Planogram Control diberi label **Academic** pada metadata repo referensi. Gunakan sumber ini untuk eksperimen riset/POC, bukan data training komersial.

In [ ]:
PUBLIC_FIELD_METADATA = []
GROCERY_PLANOGRAM_REPO = "https://github.com/meyucel/Grocery-Planogram-Control-Dataset.git"
GROCERY_DATASET_REPO = "https://github.com/gulvarol/grocerydataset.git"
GROCERY_DATASET_ARCHIVES = [
    "https://github.com/gulvarol/grocerydataset/releases/download/1.0/GroceryDataset_part1.tar.gz",
    "https://github.com/gulvarol/grocerydataset/releases/download/1.0/GroceryDataset_part2.tar.gz",
]
WIKIMEDIA_IMAGES = [
    {
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Supermarket_in_South_Korea.JPG/1280px-Supermarket_in_South_Korea.JPG",
        "filename": "wm_supermarket_south_korea.jpg",
        "source_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6d/Supermarket_in_South_Korea.JPG/1280px-Supermarket_in_South_Korea.JPG",
        "license": "CC BY-SA",
        "description": "South Korean supermarket shelf",
    },
    {
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/12/Grocery_store%2C_type_A%2C_Volkovo_%28Leningrad_Oblast%29%2C_Russia_%282022%29.jpg/1280px-Grocery_store%2C_type_A%2C_Volkovo_%28Leningrad_Oblast%29%2C_Russia_%282022%29.jpg",
        "filename": "wm_grocery_russia.jpg",
        "source_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/1/12/Grocery_store%2C_type_A%2C_Volkovo_%28Leningrad_Oblast%29%2C_Russia_%282022%29.jpg/1280px-Grocery_store%2C_type_A%2C_Volkovo_%28Leningrad_Oblast%29%2C_Russia_%282022%29.jpg",
        "license": "CC BY-SA",
        "description": "Russian grocery store shelf",
    },
    {
        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/Supermarket_shelf_in_Koog_aan_de_Zaan.jpg/1280px-Supermarket_shelf_in_Koog_aan_de_Zaan.jpg",
        "filename": "wm_supermarket_netherlands.jpg",
        "source_url": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/Supermarket_shelf_in_Koog_aan_de_Zaan.jpg/1280px-Supermarket_shelf_in_Koog_aan_de_Zaan.jpg",
        "license": "CC BY-SA",
        "description": "Dutch supermarket shelf",
    },
]

def download_file(url: str, destination: Path, timeout: int = 120) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and destination.stat().st_size > 0:
        return True
    try:
        with requests.get(url, timeout=timeout, stream=True) as response:
            response.raise_for_status()
            with destination.open("wb") as output_file:
                shutil.copyfileobj(response.raw, output_file)
        return True
    except Exception as exc:
        print(f"Gagal mengunduh {url}: {exc}")
        return False

def copy_public_image(source_path: Path, images_dir: Path, filename: str, metadata: dict) -> bool:
    images_dir.mkdir(parents=True, exist_ok=True)
    destination = images_dir / filename
    if not destination.exists():
        shutil.copy2(source_path, destination)
    PUBLIC_FIELD_METADATA.append({"filename": destination.name, **metadata})
    return True

def safe_extract_tar(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    destination_root = destination.resolve()
    with tarfile.open(archive_path, "r:gz") as archive:
        for member in archive.getmembers():
            member_target = (destination / member.name).resolve()
            try:
                member_target.relative_to(destination_root)
            except ValueError as exc:
                raise RuntimeError(f"Path arsip tidak aman: {member.name}") from exc
        archive.extractall(destination)

def download_grocery_planogram_control(output_dir: Path) -> int:
    images_dir = output_dir / "images"
    count = 0
    with tempfile.TemporaryDirectory() as tmp_dir:
        repo_dir = Path(tmp_dir) / "grocery_planogram_control"
        subprocess.run(["git", "clone", "--depth", "1", GROCERY_PLANOGRAM_REPO, str(repo_dir)], check=True)
        for image_path in sorted(repo_dir.rglob("*")):
            if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            filename = f"gpc_{image_path.stem}{image_path.suffix.lower()}"
            copy_public_image(
                image_path,
                images_dir,
                filename,
                {
                    "source": "Grocery Planogram Control Dataset",
                    "source_url": "https://github.com/meyucel/Grocery-Planogram-Control-Dataset",
                    "license": "Academic",
                    "usage": "research/POC review recommended",
                    "camera": "Samsung S10 Plus",
                    "difficulty": "occlusion, multi-angle, stacked products",
                },
            )
            count += 1
    return count

def download_grocery_dataset(output_dir: Path) -> int:
    images_dir = output_dir / "images"
    downloads_dir = output_dir / "_downloads"
    extract_dir = output_dir / "_grocery_dataset_extract"
    downloads_dir.mkdir(parents=True, exist_ok=True)
    count = 0

    with tempfile.TemporaryDirectory() as tmp_dir:
        metadata_repo = Path(tmp_dir) / "grocerydataset"
        subprocess.run(["git", "clone", "--depth", "1", GROCERY_DATASET_REPO, str(metadata_repo)], check=True)
        shutil.copy2(metadata_repo / "annotation.txt", output_dir / "grocerydataset_annotation.txt")
        shutil.copy2(metadata_repo / "annotations.csv", output_dir / "grocerydataset_annotations.csv")

    for archive_url in GROCERY_DATASET_ARCHIVES:
        archive_path = downloads_dir / archive_url.rsplit("/", 1)[-1]
        if download_file(archive_url, archive_path, timeout=600):
            safe_extract_tar(archive_path, extract_dir)

    shelf_images = sorted(
        path
        for path in extract_dir.rglob("*")
        if "ShelfImages" in path.parts and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    for image_path in shelf_images:
        filename = f"grocery_{image_path.stem}{image_path.suffix.lower()}"
        copy_public_image(
            image_path,
            images_dir,
            filename,
            {
                "source": "Grocery Dataset",
                "source_url": "https://github.com/gulvarol/grocerydataset",
                "license": "research only, not commercial use",
                "usage": "qualitative field evaluation only",
                "difficulty": "grocery shelf images with planogram IDs",
            },
        )
        count += 1
    return count

def download_wikimedia_shelves(output_dir: Path) -> int:
    images_dir = output_dir / "images"
    count = 0
    for item in WIKIMEDIA_IMAGES:
        destination = images_dir / item["filename"]
        if not download_file(item["url"], destination):
            continue
        PUBLIC_FIELD_METADATA.append(
            {
                "filename": destination.name,
                "source": "Wikimedia Commons",
                "source_url": item["source_url"],
                "license": item["license"],
                "usage": "qualitative field evaluation",
                "description": item["description"],
            }
        )
        count += 1
    return count

def download_public_field_sources(output_dir: Path) -> dict:
    output_dir.mkdir(parents=True, exist_ok=True)
    PUBLIC_FIELD_METADATA.clear()
    counts = {
        "grocery_planogram_control": download_grocery_planogram_control(output_dir),
        "wikimedia_commons": download_wikimedia_shelves(output_dir),
    }
    if INCLUDE_GROCERY_DATASET:
        counts["grocery_dataset"] = download_grocery_dataset(output_dir)
    metadata_path = output_dir / "metadata.json"
    metadata_report = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "counts": counts,
        "images": PUBLIC_FIELD_METADATA,
    }
    metadata_path.write_text(json.dumps(metadata_report, indent=2))
    print(json.dumps(counts, indent=2))
    print(f"Metadata sumber lapangan disimpan: {metadata_path}")
    return metadata_report

if DOWNLOAD_PUBLIC_FIELD_SOURCES:
    field_metadata = download_public_field_sources(FIELD_DATA_DIR)
else:
    print("Download sumber lapangan dilewati. Ubah DOWNLOAD_PUBLIC_FIELD_SOURCES=True untuk menjalankannya.")

## 10. Evaluasi sumber lapangan (opsional)

Bagian ini menjalankan inferensi pada semua gambar publik yang sudah dikumpulkan. Karena sumber lapangan tidak memiliki format anotasi yang sama dengan SKU-110K, hasilnya bersifat kualitatif: jumlah detection, confidence, latency, JSON per gambar, dan gambar beranotasi.

In [ ]:
def result_to_detections(result) -> list[dict]:
    detections = []
    if result.boxes is None:
        return detections
    boxes_xyxy = result.boxes.xyxy.cpu().tolist()
    confidences = result.boxes.conf.cpu().tolist()
    class_ids = result.boxes.cls.cpu().tolist()
    for bbox, confidence, class_id_value in zip(boxes_xyxy, confidences, class_ids):
        class_id = int(class_id_value)
        detections.append(
            {
                "bbox_xyxy": [round(float(value), 2) for value in bbox],
                "confidence": round(float(confidence), 6),
                "class_id": class_id,
                "class_name": result.names.get(class_id, str(class_id)),
            }
        )
    return detections

def load_field_metadata(data_dir: Path) -> list[dict]:
    metadata_path = data_dir / "metadata.json"
    if not metadata_path.is_file():
        raise FileNotFoundError(
            f"Metadata sumber lapangan tidak ditemukan: {metadata_path}. "
            "Jalankan DOWNLOAD_PUBLIC_FIELD_SOURCES=True terlebih dahulu."
        )
    metadata = json.loads(metadata_path.read_text())
    return metadata.get("images", [])

if RUN_FIELD_SOURCE_EVALUATION:
    field_records = load_field_metadata(FIELD_DATA_DIR)
    if FIELD_MAX_IMAGES > 0:
        field_records = field_records[:FIELD_MAX_IMAGES]
    if not field_records:
        raise RuntimeError("Tidak ada gambar sumber lapangan untuk dievaluasi.")

    annotated_dir = FIELD_OUTPUT_DIR / "annotated"
    reports_dir = FIELD_OUTPUT_DIR / "json"
    annotated_dir.mkdir(parents=True, exist_ok=True)
    reports_dir.mkdir(parents=True, exist_ok=True)
    field_reports = []

    for record in field_records:
        field_image_path = FIELD_DATA_DIR / "images" / record["filename"]
        if not field_image_path.is_file():
            print(f"Lewati file yang tidak ditemukan: {field_image_path}")
            continue

        started = time.perf_counter()
        field_results = model.predict(
            source=str(field_image_path),
            conf=CONFIDENCE,
            iou=IOU_THRESHOLD,
            imgsz=IMAGE_SIZE,
            device=DEVICE,
            verbose=False,
        )
        elapsed_ms = (time.perf_counter() - started) * 1000
        field_result = field_results[0]
        field_detections = result_to_detections(field_result)
        annotated = field_result.plot()
        annotated_path = annotated_dir / f"{field_image_path.stem}_annotated.jpg"
        cv2.imwrite(str(annotated_path), annotated)

        image_report = {
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "model_sha256": model_sha256,
            "source_metadata": record,
            "image_path": str(field_image_path),
            "annotated_path": str(annotated_path),
            "confidence_threshold": CONFIDENCE,
            "iou_threshold": IOU_THRESHOLD,
            "image_size": IMAGE_SIZE,
            "device": str(DEVICE),
            "wall_clock_ms": round(elapsed_ms, 3),
            "yolo_speed_ms": {
                key: round(float(value), 3)
                for key, value in field_result.speed.items()
            },
            "detection_count": len(field_detections),
            "detections": field_detections,
        }
        report_path = reports_dir / f"{field_image_path.stem}_result.json"
        report_path.write_text(json.dumps(image_report, indent=2))
        field_reports.append(image_report)
        print(f"{record['filename']}: {len(field_detections)} detections, {elapsed_ms:.1f} ms")

    summary = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "model_sha256": model_sha256,
        "image_count": len(field_reports),
        "sources": sorted({report["source_metadata"].get("source", "unknown") for report in field_reports}),
        "reports": field_reports,
    }
    summary_path = FIELD_OUTPUT_DIR / "field_source_results.json"
    summary_path.write_text(json.dumps(summary, indent=2))
    print(f"Ringkasan evaluasi sumber lapangan disimpan: {summary_path}")
else:
    print("Evaluasi sumber lapangan dilewati. Ubah RUN_FIELD_SOURCE_EVALUATION=True untuk menjalankannya.")

## Interpretasi hasil POC

POC awal dianggap berhasil bila model dapat dimuat, bounding box masuk akal secara visual, hasil JSON terbentuk, dan latency tercatat. Jangan menilai model hanya dari satu gambar.

Untuk kandidat MVP, gunakan SKU-110K test split sebagai evaluasi kuantitatif dan sumber lapangan publik sebagai evaluasi kualitatif. Nilai berikut dapat menjadi titik awal, bukan aturan mutlak:

- mAP@50 minimal 0.80.
- mAP@50-95 minimal 0.50.
- Precision minimal 0.85.
- Recall minimal 0.88.

Periksa juga foto blur, gelap, miring, produk kecil, occlusion, produk bertumpuk, dan rak sangat padat. Detector ini hanya mengenali keberadaan objek produk, belum mengenali identitas SKU.